In [2]:
print("RAG HR Agent")

RAG HR Agent


In [1]:
import os
from dotenv import load_dotenv

#Langchain libraries for loading
from langchain_community.document_loaders import TextLoader

#Spllitting data
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Embeddings
from langchain_community.embeddings import JinaEmbeddings

#Vector DB
from langchain_community.vectorstores import FAISS

#LLM
from langchain_groq import ChatGroq

C:\Users\mishr\AppData\Local\Temp\ipykernel_18504\4162995331.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [6]:
source_data_path = os.path.join("data", "hr_policy.txt")

In [7]:
text_data_loader = TextLoader(source_data_path, encoding="utf-8")

#Langchain process in documents
documents = text_data_loader.load()

print("="*40)
print("Data loaded")
print(documents)


Data loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

In [8]:
print(documents[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leav

In [9]:
total_lengthof_char = len(documents[0].page_content)
print(total_lengthof_char)

2597


In [10]:
#Splitting our data
text_data_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 50
)

text_chunk = text_data_splitter.split_documents(documents)
print(text_chunk)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='A medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nE

In [11]:
print(text_chunk[8].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [20]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

vector_store = FAISS.from_documents(text_chunk, embeddings_model)

print("Chunks stored", vector_store.index.ntotal)



Chunks stored 10


In [21]:
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query, k=2 )

for i, match in enumerate(top_matches, start=1):
    print(f"-----Match{i}-----")
    print(match.page_content)

-----Match1-----
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
-----Match2-----
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [3]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0
)

print(llm.model_name)

openai/gpt-oss-120b


In [4]:
test_response = llm.invoke("Can you tell me top 3 best LLMs as of today?")

In [8]:
print(test_response.content)

Sure! As of August 2026, the three large‑language models that are most widely regarded as the “best” (in terms of overall performance, versatility, and industry adoption) are:

| Rank | Model | Developer / Owner | Key Strengths | Typical Use‑Cases |
|------|-------|-------------------|----------------|-------------------|
| **1** | **GPT‑4o (Omni)** | **OpenAI** | • State‑of‑the‑art performance on a wide range of benchmarks (reasoning, coding, multilingual, multimodal). <br>• Supports text, images, audio, and video inputs with seamless context‑switching. <br>• Strong safety and alignment layers (RLHF + Constitutional AI). | • Enterprise assistants, research assistants, creative generation, multimodal content creation, code generation, tutoring. |
| **2** | **Claude 3.5 Sonnet** | **Anthropic** | • Consistently top‑ranked on reasoning and instruction‑following tests (e.g., MMLU, BIG‑Bench). <br>• Emphasis on “constitutional” safety, making it very reliable for high‑stakes applications. 